## This notebook is trying to register a low res DESI and high RES Xenium OMI TIFF images together

In [1]:
import warnings
warnings.filterwarnings("ignore")

import spatialdata as sd
from spatialdata_io import xenium
from pathlib import Path
from spatialdata.models import Image2DModel
from spatialdata.transformations import Scale
import tifffile
from spatialdata.models import get_channel_names
from napari_spatialdata import Interactive
import napari
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

In [5]:
user_home = Path.home()

data_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" 
# Define the specific project folder
xenium_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "raw" / "xenium"
msi_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "raw" /"msi"

raw_data_path = xenium_project_dir / "output-XETG00169__0055588__55588_region_4__20250418__182706"
desi_img_data_path = msi_project_dir/ "3D_DESI_F5_Pos mode_40um_F5_5pos_40um_Features110425.ome.tif"
desi_flipped_img_data_path = data_dir/ "02_channel_extraction"/ "desi_flipped.ome.tif"

In [6]:
raw_data_path

PosixPath('/Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/raw/xenium/output-XETG00169__0055588__55588_region_4__20250418__182706')

In [7]:
sdata = xenium(raw_data_path)

INFO     reading                                                                                                   
         /Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/raw/xenium/output-XETG00169__0055588__5
         5588_region_4__20250418__182706/cell_feature_matrix.h5                                                    


In [8]:
sdata

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 20495, 19973), (5, 10247, 9986), (5, 5123, 4993), (5, 2561, 2496), (5, 1280, 1248)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
│     └── 'nucleus_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (52665, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (52665, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (48131, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (52665, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), cell_circles (Shapes), nucleus_boundaries (Shapes)

In [9]:
desi_img = tifffile.imread(desi_img_data_path)

In [10]:
with tifffile.TiffFile(desi_img_data_path) as tif:
    ome_xml = tif.ome_metadata  # returns the OME-XML string
    
root = ET.fromstring(ome_xml)

# OME-XML uses a namespace
ns = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}

channels = root.findall('.//ome:Channel', ns)
channel_names = [ch.get('Name') for ch in channels]

In [11]:
channel_names

['104.11352 m/z ± 60 ppm',
 '156.05072 m/z ± 60 ppm',
 '162.12311 m/z ± 60 ppm',
 '204.13479 m/z ± 60 ppm',
 '257.16169 m/z ± 60 ppm',
 '258.12141 m/z ± 50 ppm',
 '296.08002 m/z ± 50 ppm',
 '298.07787 m/z ± 60 ppm',
 '308.10933 m/z ± 60 ppm',
 '330.08643 m/z ± 60 ppm',
 '348.08769 m/z ± 60 ppm',
 '369.37202 m/z ± 60 ppm',
 '398.34926 m/z ± 60 ppm',
 '400.36601 m/z ± 50 ppm',
 '422.34776 m/z ± 60 ppm',
 '424.36928 m/z ± 60 ppm',
 '426.37917 m/z ± 50 ppm',
 '616.25034 m/z ± 60 ppm',
 '631.50333 m/z ± 50 ppm',
 '633.51154 m/z ± 50 ppm',
 '657.52195 m/z ± 50 ppm',
 '659.53044 m/z ± 50 ppm',
 '703.61266 m/z ± 50 ppm',
 '731.65049 m/z ± 60 ppm',
 '732.60225 m/z ± 60 ppm',
 '734.60131 m/z ± 50 ppm',
 '741.57618 m/z ± 60 ppm',
 '754.57363 m/z ± 60 ppm',
 '756.59481 m/z ± 50 ppm',
 '758.60244 m/z ± 50 ppm',
 '762.6337 m/z ± 60 ppm',
 '764.56178 m/z ± 60 ppm',
 '766.57139 m/z ± 60 ppm',
 '768.58628 m/z ± 60 ppm',
 '770.54867 m/z ± 50 ppm',
 '772.57401 m/z ± 50 ppm',
 '774.60468 m/z ± 60 ppm',
 '

In [ ]:
desi_img.shape

(88, 107, 102)

In [9]:
# print("\n\n=== Finding DESI Channels ===")
# print("We need to find channels for m/z: 616.25, 772.57, 893.75")
# print("\nLet's look at some channels to identify patterns...")

# # Quick visualization of several channels to find the right ones
# viewer = napari.Viewer()

# # Add a few DESI channels to explore
# # Start with some spread across the range
# test_channels = [3,17,23,25]

# for i in test_channels:
#     viewer.add_image(
#         desi_img[i], 
#         name=f"DESI_channel_{i}",
#         visible=(i == 0),  # Only first one visible by default
#         colormap='viridis'
#     )

# print(f"\nOpened napari with channels: {test_channels}")
# print("Toggle through channels to find ones with clear tissue structure")
# print("Note which channel numbers show the morphology you saw in QuPath")

# napari.run()

In [10]:
channel_indices = [3,17,23,25]
desi_selected = desi_img[channel_indices, :, :] 

In [11]:
# Create SpatialData image with proper coordinate system
desi_spatial = Image2DModel.parse(
    desi_img,
    dims=("c", "y", "x"),
    transformations={
        "global": Scale(
            [40.0, 40.0],  # 40 μm per pixel
            axes=("y", "x")
        )
    },
    c_coords=channel_names
)

In [12]:
# Add to spatialdata object
sdata.images["desi"] = desi_spatial

In [13]:
print("\nDESI image added to SpatialData object!")
print(f"Available images now: {list(sdata.images.keys())}")


DESI image added to SpatialData object!
Available images now: ['morphology_focus', 'desi']


In [14]:
sdata.images['desi']

<xarray.DataArray 'image' (c: 88, y: 107, x: 102)> Size: 2MB
dask.array<array, shape=(88, 107, 102), dtype=uint16, chunksize=(88, 107, 102), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U22 8kB '104.11352 m/z ± 60 ppm' ... '760.6154 m/z ± 60 ppm'
  * y        (y) float64 856B 0.5 1.5 2.5 3.5 4.5 ... 103.5 104.5 105.5 106.5
  * x        (x) float64 816B 0.5 1.5 2.5 3.5 4.5 ... 97.5 98.5 99.5 100.5 101.5
Attributes:
    transform:  {'global': Scale (y, x)\n    [40. 40.]}

In [15]:
sdata.write(xenium_project_dir/ "processed"/ "sdata_object.zarr",overwrite=True)

INFO     The Zarr backing store has been changed from None the new file path:                                      
         /Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/processed/sdata_object.zarr     


In [16]:
sdata = sd.read_zarr(xenium_project_dir/ "processed"/ "sdata_object.zarr")

version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


### Using Napari-Spatial Data to visualize

In [19]:
# interactive = Interactive(sdata)
# interactive.run()

In [20]:
# 1. Create Xenium reference - combine vessel + membrane markers
morph = sdata.images['morphology_focus']

channel_names_xenium = get_channel_names(morph)
channel_names_xenium

[np.str_('DAPI'),
 np.str_('ATP1A1/CD45/E-Cadherin'),
 np.str_('18S'),
 np.str_('AlphaSMA/Vimentin'),
 np.str_('dummy')]

In [21]:
# The full resolution data is typically at 'scale0'
morph_full = morph['scale0'].to_dataset()

In [22]:
print(f"Spatial dimensions: y={morph_full.dims['y']}, x={morph_full.dims['x']}")

Spatial dimensions: y=20495, x=19973


In [23]:
# The actual array is accessed via the data variable (usually called something like 'image' or the dataset name)
# Let's find the data variable name:
print(f"\nData variables: {list(morph_full.data_vars)}")


Data variables: ['image']


In [24]:
morph_array = morph_full['image']

# Extract relevant channels in scale0 which is the highest resolution we have
# Adjust indices based on actual channel order in your file
alphasma_vim = morph_array.sel(c='AlphaSMA/Vimentin').values  # Vessel marker
atp1a1 = morph_array.sel(c='ATP1A1/CD45/E-Cadherin').values  # Membrane marker
dapi = morph_array.sel(c='DAPI').values  # Nuclei marker

In [28]:
## similar to above extract the DESI channels as well
desi = sdata.images['desi']
desi_flipped = np.flip(desi,axis=2)
mz_204 = desi_flipped.sel(c=channel_names[channel_indices[0]]).values
heme_b = desi_flipped.sel(c=channel_names[channel_indices[1]]).values
pe_ps_731 = desi_flipped.sel(c=channel_names[channel_indices[2]]).values
pe_ps_734 = desi_flipped.sel(c=channel_names[channel_indices[3]]).values

In [29]:
## saving the flipped desi image (all 88 channels)
desi_img_flipped_all = np.flip(desi_img, axis=2)
tifffile.imwrite(desi_flipped_img_data_path, desi_img_flipped_all)
print(f"Saved flipped DESI image: {desi_img_flipped_all.shape} (all {desi_img_flipped_all.shape[0]} channels)")

Saved flipped DESI image: (88, 107, 102) (all 88 channels)


In [26]:
# viewer = napari.Viewer()

# viewer.add_image(
#     atp1a1,
#     name='ATP1A1 (membranes)',
#     colormap='green',
#     scale=[0.2125, 0.2125],
#     blending='additive',
#     visible=True
# )

# viewer.add_image(
#     alphasma_vim,
#     name='AlphaSMA/Vimentin (vessels)',
#     colormap='magenta',
#     scale=[0.2125, 0.2125],
#     blending='additive',
#     visible=True
# )

# viewer.add_image(
#     dapi,
#     name='DAPI (nuclei)',
#     colormap='blue',
#     scale=[0.2125, 0.2125],  # Xenium pixel size
#     blending='additive',
#     visible=True
# )

# # DESI channels at 40 μm/pixel
# viewer.add_image(
#     mz_204,
#     name='DESI mz_204.13',
#     colormap='cyan',
#     scale=[40.0, 40.0],``
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     heme_b,
#     name='DESI Heme B (616.25)',
#     colormap='yellow',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     pe_ps_731,
#     name='DESI PE/PS (731.65)',
#     colormap='red',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     pe_ps_734,
#     name='DESI PE/PS (734.60)',
#     colormap='bop orange',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

In [30]:
# Normalize each channel to [0, 1]
def normalize(img):
    img = img.astype(float)
    return (img - img.min()) / (img.max() - img.min() + 1e-10)

In [31]:
# Xenium composite
xenium_composite = np.stack([
    normalize(alphasma_vim),  # Red - vessels
    normalize(atp1a1),        # Green - membranes
    np.zeros_like(atp1a1)     # Blue - empty
], axis=-1)

# DESI composite
desi_composite = np.stack([
    normalize(heme_b),      # Red - vessels (matches Xenium red)
    normalize(pe_ps_731), # Green - membranes (matches Xenium green)
    np.zeros_like(heme_b)   # Blue - empty
], axis=-1)

print("\n=== Composite Images Created ===")
print("Red channel: Vessels (AlphaSMA ↔ Heme B)")
print("Green channel: Membranes (ATP1A1 ↔ Lipid 731.65)")


=== Composite Images Created ===
Red channel: Vessels (AlphaSMA ↔ Heme B)
Green channel: Membranes (ATP1A1 ↔ Lipid 731.65)


In [32]:
# viewer = napari.Viewer()

# # Add Xenium composite (FIXED - reference)
# viewer.add_image(
#     xenium_composite,
#     name='Xenium_Composite (FIXED)',
#     rgb=True,
#     scale=[0.2125, 0.2125]
# )

# # Add DESI composite (MOVING - to be registered)
# viewer.add_image(
#     desi_composite,
#     name='DESI_Composite (MOVING)',
#     rgb=True,
#     scale=[40.0, 40.0],
#     opacity=0.7,
#     blending='additive'
# )

# # Optional: Add individual channels for reference
# viewer.add_image(
#     alphasma_vim,
#     name='Xenium_AlphaSMA (vessels only)',
#     colormap='red',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     atp1a1,
#     name='Xenium_ATP1A1 (membranes only)',
#     colormap='green',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     heme_b,
#     name='DESI_HemeB (vessels only)',
#     colormap='red',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# viewer.add_image(
#     pe_ps_731,
#     name='DESI_Lipid731 (membranes only)',
#     colormap='green',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# # Add points layers for landmarks
# # border_width is relative (0-1) by default in this napari version
# fixed_points = viewer.add_points(
#     name='Xenium_landmarks (FIXED)',
#     ndim=2,
#     face_color='cyan',
#     size=200,
#     border_color='white',
#     border_width=0.2,
#     opacity=0.9
# )

# moving_points = viewer.add_points(
#     name='DESI_landmarks (MOVING)',
#     ndim=2,
#     face_color='yellow',
#     size=200,
#     border_color='white',
#     border_width=0.2,
#     opacity=0.9
# )

# napari.run()

### Loading the Landmarks Created in QuPATH

In [33]:
moving_landmarks_path = data_dir /"3D_DESI_F5_Pos mode_40um_Flipped.ome.tif - Image0-points.tsv"
fixed_landmarks_path = data_dir /"morphology_focus_0002.ome.tif-points.tsv"

In [34]:
xenium_df = pd.read_csv(fixed_landmarks_path, sep='\t')
desi_df = pd.read_csv(moving_landmarks_path, sep='\t')

In [35]:
# Extract point number from the class column (last part after '_')
xenium_df['point_num'] = xenium_df['class'].str.extract(r'_(\d+)$').astype(int)
desi_df['point_num'] = desi_df['class'].str.extract(r'_(\d+)$').astype(int)

# Add coordinates in microns
xenium_df['x_microns'] = xenium_df['x'] * 0.2125
xenium_df['y_microns'] = xenium_df['y'] * 0.2125
desi_df['x_microns'] = desi_df['x'] * 40.0
desi_df['y_microns'] = desi_df['y'] * 40.0

# Join on point number
landmarks_df = xenium_df[['point_num', 'x', 'y', 'x_microns', 'y_microns']].merge(
    desi_df[['point_num', 'x', 'y', 'x_microns', 'y_microns']],
    on='point_num',
    suffixes=('_xenium', '_desi')
).sort_values('point_num').reset_index(drop=True)

landmarks_df

,point_num,x_xenium,y_xenium,x_microns_xenium,y_microns_xenium,x_desi,y_desi,x_microns_desi,y_microns_desi
0,1,402.770996,457.542877,85.588837,97.227861,6.131444,1.119414,245.257778,44.776578
1,2,3699.857910,3422.365234,786.219806,727.252612,22.170193,16.869177,886.807709,674.767075
2,3,16325.912109,15051.625000,3469.256323,3198.470313,92.971878,80.301704,3718.875122,3212.068176
3,4,15124.647461,11907.891602,3213.987585,2530.426965,86.180695,63.829479,3447.227783,2553.179169
4,5,8377.121094,5697.099609,1780.138232,1210.633667,45.722591,30.740528,1828.903656,1229.621124
5,6,8351.562500,13748.125977,1774.707031,2921.476770,52.224785,75.677925,2088.991394,3027.117004
6,7,2247.153320,14362.148438,477.520081,3051.956543,16.899088,78.816422,675.963516,3152.656860
7,8,722.773193,4879.209961,153.589304,1036.832117,5.398269,20.834419,215.930748,833.376770
8,9,837.363770,7197.788086,177.939801,1529.529968,5.559475,37.277485,222.378998,1491.099396
9,10,4642.672363,17427.021484,986.567877,3703.242065,27.646935,94.320389,1105.877380,3772.815552


In [36]:
## dropping some points to make the transformation better
# landmarks_df = landmarks_df.iloc[[0,1,3]]

### Using SimpleITK to create an affine transform

In [37]:
import SimpleITK as sitk

# Convert to SimpleITK images (use AlphaSMA as the Xenium reference, Heme B as DESI reference)
xenium_sitk = sitk.GetImageFromArray(alphasma_vim)
xenium_sitk.SetSpacing([0.2125, 0.2125])  # x, y spacing in microns
xenium_sitk.SetOrigin([0.0, 0.0])

desi_sitk = sitk.GetImageFromArray(heme_b)
desi_sitk.SetSpacing([40.0, 40.0])  # x, y spacing in microns
desi_sitk.SetOrigin([0.0, 0.0])

print(f"Xenium SimpleITK: size={xenium_sitk.GetSize()}, spacing={xenium_sitk.GetSpacing()}")
print(f"DESI SimpleITK: size={desi_sitk.GetSize()}, spacing={desi_sitk.GetSpacing()}")

# Prepare landmarks for SimpleITK — list of [x, y] in physical coordinates (microns)
fixed_landmarks = [[float(row.x_microns_xenium), float(row.y_microns_xenium)] for _, row in landmarks_df.iterrows()]
moving_landmarks = [[float(row.x_microns_desi), float(row.y_microns_desi)] for _, row in landmarks_df.iterrows()]

print(f"\nFixed landmarks (Xenium, microns): {fixed_landmarks}")
print(f"Moving landmarks (DESI, microns): {moving_landmarks}")

Xenium SimpleITK: size=(19973, 20495), spacing=(0.2125, 0.2125)
DESI SimpleITK: size=(102, 107), spacing=(40.0, 40.0)

Fixed landmarks (Xenium, microns): [[85.58883666992188, 97.22786140441895], [786.219805908203, 727.2526123046874], [3469.2563232421876, 3198.4703125], [3213.987585449219, 2530.426965332031], [1780.138232421875, 1210.6336669921875], [1774.70703125, 2921.476770019531], [477.52008056640625, 3051.95654296875], [153.58930358886718, 1036.8321166992187], [177.9398010253906, 1529.5299682617188], [986.5678771972656, 3703.2420654296875], [1205.2582885742188, 3849.801904296875], [3574.68828125, 595.9207641601562]]
Moving landmarks (DESI, microns): [[245.2577781677246, 44.77657794952392], [886.8077087402344, 674.7670745849609], [3718.875122070312, 3212.0681762695312], [3447.227783203125, 2553.179168701172], [1828.9036560058591, 1229.6211242675781], [2088.9913940429688, 3027.1170043945312], [675.9635162353516, 3152.6568603515625], [215.93074798583984, 833.3767700195312], [222.37899

In [38]:
# Initialize affine transform
transform = sitk.AffineTransform(2)  # 2D affine

# SimpleITK expects landmarks as a flat list: [x1, y1, x2, y2, ...]
fixed_landmarks_flat = [coord for pt in fixed_landmarks for coord in pt]
moving_landmarks_flat = [coord for pt in moving_landmarks for coord in pt]

# Compute transformation from landmarks
transform = sitk.LandmarkBasedTransformInitializer(
    transform,
    fixed_landmarks_flat,
    moving_landmarks_flat
)

print("Affine transformation computed")

# Extract transformation parameters
matrix = transform.GetMatrix()
translation = transform.GetTranslation()

print(f"\nTransformation Matrix:")
print(f"  [{matrix[0]:8.5f}  {matrix[1]:8.5f}]")
print(f"  [{matrix[2]:8.5f}  {matrix[3]:8.5f}]")

print(f"\nTranslation: [{translation[0]:.2f}, {translation[1]:.2f}] um")

# Compute derived parameters
rotation_rad = np.arctan2(matrix[2], matrix[0])
rotation_deg = np.degrees(rotation_rad)
scale_x = np.sqrt(matrix[0]**2 + matrix[2]**2)
scale_y = np.sqrt(matrix[1]**2 + matrix[3]**2)

print(f"\nDerived Parameters:")
print(f"  Rotation: {rotation_deg:.2f} degrees")
print(f"  Scale X: {scale_x:.4f}")
print(f"  Scale Y: {scale_y:.4f}")

Affine transformation computed

Transformation Matrix:
  [ 1.03663   0.01734]
  [-0.00476   1.06145]

Translation: [67.90, -116.42] um

Derived Parameters:
  Rotation: -0.26 degrees
  Scale X: 1.0366
  Scale Y: 1.0616


In [39]:
# Transform the moving landmarks and compute errors
transformed_landmarks = []
for pt in moving_landmarks:
    transformed_pt = transform.TransformPoint(pt)
    transformed_landmarks.append(transformed_pt)

transformed_landmarks = np.array(transformed_landmarks)
fixed_landmarks_array = np.array(fixed_landmarks)

# Compute Euclidean distance errors
errors = np.linalg.norm(transformed_landmarks - fixed_landmarks_array, axis=1)

print(f"\nLandmark Registration Errors (µm):")
for i, (name, error) in enumerate(zip(xenium_df['class'], errors)):
    print(f"  {name}: {error:.2f} µm")

print(f"\nSummary Statistics:")
print(f"  Mean error:   {errors.mean():.2f} µm")
print(f"  Median error: {np.median(errors):.2f} µm")
print(f"  Max error:    {errors.max():.2f} µm")
print(f"  Min error:    {errors.min():.2f} µm")
print(f"  Std dev:      {errors.std():.2f} µm")

# Quality assessment
if errors.mean() < 50:
    print("\n✓✓✓ EXCELLENT registration (mean < 50 µm)")
elif errors.mean() < 100:
    print("\n✓✓ GOOD registration (mean < 100 µm)")
elif errors.mean() < 200:
    print("\n✓ FAIR registration (mean < 200 µm)")
else:
    print("\n⚠ WARNING: Poor registration (mean > 200 µm)")
    print("  Consider re-placing landmarks")


Landmark Registration Errors (µm):
  Xenium_Point_3: 290.36 µm
  Xenium_Point_6: 250.12 µm
  Xenium_Point_11: 515.18 µm
  Xenium_Point_5: 473.98 µm
  Xenium_Point_1: 207.24 µm
  Xenium_Point_12: 537.23 µm
  Xenium_Point_8: 387.42 µm
  Xenium_Point_2: 309.87 µm
  Xenium_Point_9: 159.83 µm
  Xenium_Point_4: 343.83 µm
  Xenium_Point_7: 424.15 µm
  Xenium_Point_10: 483.60 µm

Summary Statistics:
  Mean error:   365.24 µm
  Median error: 365.63 µm
  Max error:    537.23 µm
  Min error:    159.83 µm
  Std dev:      119.51 µm

⚠ WARNING: Poor registration (mean > 200 µm)
  Consider re-placing landmarks


In [40]:
# Resample DESI image to Xenium coordinate space
desi_registered_sitk = sitk.Resample(
    desi_sitk,           # Moving image
    xenium_sitk,         # Reference image (defines output space)
    transform,           # Transformation
    sitk.sitkLinear,     # Interpolation method
    0.0,                 # Default pixel value
    desi_sitk.GetPixelID()  # Output pixel type
)

# Convert back to numpy for visualization
desi_registered = sitk.GetArrayFromImage(desi_registered_sitk)

print(f"✓ DESI image registered to Xenium space")
print(f"  Registered image shape: {desi_registered.shape}")
print(f"  Now in same coordinate system as Xenium!")
print(f"  Pixel spacing: {desi_registered_sitk.GetSpacing()} µm")

✓ DESI image registered to Xenium space
  Registered image shape: (20495, 19973)
  Now in same coordinate system as Xenium!
  Pixel spacing: (0.2125, 0.2125) µm


In [41]:
# ## visulizing the registered image in Napari

# viewer = napari.Viewer()

# viewer.add_image(
#     alphasma_vim,
#     name='Xenium_AlphaSMA (vessels only)',
#     colormap='red',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     atp1a1,
#     name='Xenium_ATP1A1 (membranes only)',
#     colormap='green',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     desi_registered,
#     name='DESI_HemeB_REGISTERED',
#     colormap='red',
#     scale=[0.2125, 0.2125],  # Same spacing as Xenium now!
#     blending='additive',
#     opacity=0.7
# )

# viewer.add_image(
#     heme_b,
#     name='DESI_HemeB (vessels only)',
#     colormap='red',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# viewer.add_image(
#     pe_ps_731,
#     name='DESI_Lipid731 (membranes only)',
#     colormap='green',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# napari.run()